# Project Overview & Objectives

 This research document outlines a production-grade script for **Domain-Specific Pretraining and Fine-Tuning** of Large Language Models (LLMs). The primary objective is to adapt a pre-trained base model to a specialized medical/pharmaceutical domain using unsupervised causal language modeling (next-token prediction), entirely without relying on instruction-response pairs.
 
**Key Phases in this Architecture:**

**Environment Setup:** Installing and configuring required libraries.

**Data Acquisition & EDA:** Fetching prebuilt data for testing and parsing custom raw PDF documents.

**Preprocessing & Feature Engineering:** Chunking texts into context windows and preparing causal language modeling targets.

**Model Development & Training:** Exploring Full Fine-Tuning, Layer Freezing, and Parameter-Efficient Fine-Tuning (LoRA).

**Evaluation & Results:** Generating text to verify domain adaptation.

**Conclusion & Future Work:** Mapping out instruction tuning and preference alignment.

# Environment Setup

## Installation of Required Libraries

In [ ]:
%pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF datasets torchao

## Importing Libraries

In [ ]:
from IPython.display import display
from datasets import load_dataset, Dataset
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import re
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch

## Mount Google Drive and create a target folder for notebook assets

In [5]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ASSETS = Path('/content/drive/MyDrive/LLM-Fine-Tuning/DomainSpecific/assets')
DRIVE_ASSETS.mkdir(parents=True, exist_ok=True)
print("Drive assets folder:", DRIVE_ASSETS)

# Data Acquisition & Exploratory Data Analysis (EDA)

## Ingesting Prebuilt Demonstration Data from HuggingFace

In [6]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

In [7]:
display(dataset[:3])

## Custom Extraction Pipeline for Domain-Specific PDFs

In [8]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [9]:
pdf_texts = extract_text_from_pdf(f"{DRIVE_ASSETS}/Metformin.pdf")
display(pdf_texts[:2])

# Preprocessing & Feature Engineering

## Text Segmentation and Cleaning

In [10]:
def split_paragraphs(pages):
    """
    Uses regex to split page text on double line breaks.
    Filters out extremely short, fragmented lines (e.g., page numbers or headers).
    """
    processed_paragraphs = []
    for page in pages:
        chunks = re.split(r'\n\s*\n', page)
        for chunk in chunks:
            clean_chunk = chunk.strip()
            if len(clean_chunk) > 50:  # Filter out very short lines
                processed_paragraphs.append(clean_chunk)
    return processed_paragraphs

In [11]:
domain_paragraphs = split_paragraphs(pdf_texts)
formatted_data = [{"text": para} for para in domain_paragraphs]
domain_dataset = Dataset.from_list(formatted_data)
display(domain_dataset[:2])

## Mapping the Tokenizer

In [12]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# Initialize tokenizer and handle padding constraints
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [13]:
def tokenize_fn(examples):
    """
    Tokenizes raw text strings into dense integer IDs, truncating 
    and padding to standard 512 context length.
    """
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    # Duplicate input_ids into labels for auto-shifted next-token prediction
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [14]:
# Map tokenization function across the entire domain dataset
tokenized_dataset = domain_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

## Dynamic Data Collator Injection

In [15]:
# Instantiating the proper language modeling collator. mlm=False means Causal (GPT-style) LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Model Development & Training

In [16]:
model = AutoModelForCausalLM.from_pretrained(model_name)

## Approach 1: Full Fine-Tuning (Resource Intensive)
 This updates all 1.1 Billion weights. While mathematically optimal, it is highly prone to catastrophic forgetting and requires immense VRAM.

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./llama-pharma-domain-full",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

trainer_full = Trainer(
    model=model,
    args=training_args_full,
    train_dataset=tokenized_dataset,
    data_collator=data_collator # Added for structural completeness
)

In [ ]:
# trainer_full.train()

## Approach 2: Partial Fine-Tuning (Layer Freezing)
This methodology locks early feature layers and only updates the final few transformer blocks along with the language model head to save computational overhead.

In [ ]:
# Step A: Freeze everything first
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Step B: Dynamically find how many layers the model has
num_layers = model.config.num_hidden_layers
unfreeze_last_n_layers = 4
start_layer = max(0, num_layers - unfreeze_last_n_layers)

print(f"Total layers: {num_layers}. Unfreezing layers {start_layer} to {num_layers - 1}...")

In [ ]:
# Step C: Unfreeze the targeted layers, the final LayerNorm, and the LM Head
for name, param in model.named_parameters():
    # 1. Unfreeze the last N transformer blocks
    if any(f"model.layers.{i}." in name for i in range(start_layer, num_layers)):
        param.requires_grad = True
    
    # 2. Unfreeze the final layer normalization (crucial for stability)
    if "model.norm" in name:
        param.requires_grad = True
        
    # 3. Unfreeze the Language Modeling Head (the output projection)
    if "lm_head" in name:
        param.requires_grad = True

In [ ]:
# Verification: Print trainable percentage
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# Training Configuration & Execution
# Enable gradient checkpointing to save memory if your GPU is small
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="./tinyllama-pharma-frozen-layers",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,  # We can use a slightly higher LR than full fine-tuning
    bf16=True,  # Change to fp16=True if your GPU doesn't support bfloat16
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

trainer_partial = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [ ]:
# trainer_partial.train()

## Approach 3: Parameter-Efficient Fine-Tuning (LoRA)
Low-Rank Adaptation (LoRA) injects trainable rank decomposition matrices into the transformer blocks while keeping the base model weights frozen. This dramatically cuts memory requirements.


In [17]:
# Clean up VRAM allocations before starting quantized loading
# del model
# gc.collect()
# torch.cuda.empty_cache()

# 1. Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    # Optional but recommended for training:
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# 2. Load the model using the config
peft_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Configure LoRA Adapter
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                  
    lora_alpha=16,        
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

# wrap the model with get_peft_model so it targets weights correctly!
non_inst_model_lora = get_peft_model(peft_model, lora_config)
non_inst_model_lora.print_trainable_parameters() # Good practice to verify LoRA is active

lora_training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,  
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

# add data_collator and pass the actual wrapped LoRA model
trainer_peft = Trainer(
    model=non_inst_model_lora,
    args=lora_training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

### Train the model and save the model

To properly save our LoRA-trained model in an optimal and professional manner, we need to understand that PEFT (Parameter-Efficient Fine-Tuning) splits saving into two core components:

- The Adapter Weights: Saving the tiny LoRA modifications (adapter_model.bin or adapter_model.safetensors and adapter_config.json).

- The Merged Model (Optional but Recommended): Merging those adapters back into the massive base model so we can use it like a traditional, independent LLM.

Since we are running Google Colab via VS Code locally, it is highly recommended to save these files directly to our persistent Google Drive or a local directory so they don't disappear when the Colab runtime disconnects.

**1. Saving the LoRA Adapters**

First, configure your Trainer or model instance to output the LoRA configurations and sparse weights. Add this immediately after trainer.train()

In [19]:
trainer_peft.train()

# --- STEP 1: SAVE THE LORA ADAPTER WEIGHTS AND TOKENIZER ---
# Define a professional path. If you want it on Google Drive, mount it and use: "/content/drive/MyDrive/tinyllama-lora-adapter"
output_adapter_dir = f"{DRIVE_ASSETS}/tinyllama-lora-adapter"

print(f"Saving LoRA adapter to {output_adapter_dir}...")
# This only saves the lightweight adapter weights (a few Megabytes)
trainer_peft.model.save_pretrained(output_adapter_dir)
tokenizer.save_pretrained(output_adapter_dir)

print("✅ Adapter saved successfully!")

**2. Merging and Saving the Full Model (The Production Approach)**

While saving the adapters is fast, it requires you to load the original base model and the adapter every time you want to do inference. Merging them into a single file is the optimal way to deploy.

Crucial VRAM Caveat: You cannot merge an adapter into an 8-bit or 4-bit quantized base model natively because quantization dynamically alters the precision of the base weights. To merge safely without running out of memory, clean your cache, reload the base model in floating-point 16, and fuse them.

In [ ]:
# Clean up any residual memory allocations before loading the large model
# gc.collect()
# torch.cuda.empty_cache()

# 1. Reload the pristine base model (unquantized) in float16 precision
print("Reloading unquantized base model back into memory...")
base_model_reload = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # FIX: Corrected argument name
    device_map="auto"
)

# 2. Re-load the adapter on top of the clean base model
print(f"Loading adapter weights from: {output_adapter_dir}")
model_to_merge = PeftModel.from_pretrained(base_model_reload, output_adapter_dir)

# 3. Mathematically fuse the weights together
print("Fusing LoRA weights with base model layers...")
merged_model = model_to_merge.merge_and_unload()

# --- STEP 3: SAVE THE FINAL CONSOLIDATED PRODUCTION MODEL ---
output_final_dir = f"{DRIVE_ASSETS}/tinyllama-pharma-production-model"
print(f"Saving fully combined production model to: {output_final_dir}")

merged_model.save_pretrained(output_final_dir)
tokenizer.save_pretrained(output_final_dir)

print("🎉 Model successfully merged and saved to your Google Drive!")

# Evaluation & Results

In [31]:
trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-lora/checkpoint-5", device_map="auto")  # active in-memory representation

# Setup prompt validation
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Text generation execution
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [34]:
print("\n--- Model Generated Output ---\n")
print(tokenizer.decode(outputs, skip_special_tokens=False))